# Step 1: Authentication Test

**Goal:** Verify that we can successfully login and receive a secret key from the server.

**What we're doing:**
1. Import necessary libraries
2. Define the base URL and student ID
3. Create a login function
4. Test the login and display the secret key

**Expected Result:**
- Status code: 200 (success)
- A 64-character SHA256 secret key

In [1]:
# Import required libraries
import requests
import json
from pprint import pprint

# Configuration
BASE_URL = "http://72.60.221.150:8080"
STUDENT_ID = "MDS202506"

print("Libraries imported successfully!")
print(f" Server URL: {BASE_URL}")
print(f" Student ID: {STUDENT_ID}")

Libraries imported successfully!
 Server URL: http://72.60.221.150:8080
 Student ID: MDS202506


In [2]:
def login(student_id):
    """
    Authenticate with the server and get a secret key.
    
    Args:
        student_id (str): Your student ID
        
    Returns:
        str: Secret key if successful, None if failed
    """
    login_url = f"{BASE_URL}/login"
    payload = {"student_id": student_id}
    
    try:
        print(f" Attempting login for student: {student_id}")
        print(f"Sending POST request to: {login_url}")
        print(f" Payload: {payload}")
        print("-" * 60)
        
        response = requests.post(login_url, json=payload)
        
        print(f" Response Status Code: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            secret_key = data.get('secret_key')
            
            print(" Login Successful!")
            print(f" Secret Key: {secret_key}")
            print(f" Key Length: {len(secret_key)} characters")
            
            return secret_key
        else:
            print(f" Login Failed!")
            print(f"Response: {response.text}")
            return None
            
    except Exception as e:
        print(f" Error occurred: {str(e)}")
        return None

In [3]:
import time

def get_publication_title(secret_key, filename):
    """
    Retrieve a publication title from the server.
    
    Args:
        secret_key (str): Your authenticated secret key
        filename (str): Publication filename (e.g., 'pub_0.txt')
        
    Returns:
        str: Publication title if successful, None if failed
    """
    lookup_url = f"{BASE_URL}/lookup"
    payload = {
        "secret_key": secret_key,
        "filename": filename
    }
    
    try:
        print(f" Requesting title for: {filename}")
        response = requests.post(lookup_url, json=payload)
        
        print(f"Response Status Code: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            title = data.get('title')
            print(f"Title Retrieved: {title}")
            return title
            
        elif response.status_code == 429:
            print(f"Rate limit hit (429) - Too many requests")
            return None
            
        elif response.status_code == 401:
            print(f"Unauthorized - Invalid secret key")
            return None
            
        elif response.status_code == 404:
            print(f"File not found: {filename}")
            return None
            
        else:
            print(f"Unexpected error: {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        return None

def extract_first_word(title):
    """
    Extract the first word from a publication title.
    
    Args:
        title (str): Full publication title
        
    Returns:
        str: First word from the title
    """
    if not title:
        return None
    
    # Remove leading/trailing whitespace
    title = title.strip()
    
    # Split by whitespace and get first word
    words = title.split()
    
    if words:
        first_word = words[0]
        
        # Remove common leading punctuation (quotes, parentheses)
        first_word = first_word.lstrip('("\'')
        
        return first_word
    
    return None

print("Functions defined successfully!")

Functions defined successfully!


In [4]:
print("=" * 60)
print("TESTING TITLE RETRIEVAL")
print("=" * 60)

# First, login to get a fresh secret key
print("\n Step 1: Login to get secret key...")
secret_key = login(STUDENT_ID)

if secret_key:
    print(f" Secret Key: {secret_key[:32]}...")
    
    print("\n Step 2: Retrieve title for pub_0.txt...")
    print("-" * 60)
    
    # Test retrieving pub_0.txt
    title = get_publication_title(secret_key, "pub_0.txt")
    
    if title:
        print("\n" + "=" * 60)
        print(" RESULTS")
        print("=" * 60)
        print(f"Filename: pub_0.txt")
        print(f"Title: {title}")
        
        # Extract first word
        first_word = extract_first_word(title)
        print(f"First Word: {first_word}")
        print("=" * 60)
        print("\n SUCCESS! Title retrieval is working!")
    else:
        print("\n Failed to retrieve title")
else:
    print("\n Login failed - cannot proceed")

TESTING TITLE RETRIEVAL

 Step 1: Login to get secret key...
 Attempting login for student: MDS202506
Sending POST request to: http://72.60.221.150:8080/login
 Payload: {'student_id': 'MDS202506'}
------------------------------------------------------------
 Response Status Code: 200
 Login Successful!
 Secret Key: 5a041cabf258ed6c15d4b04fd8d9cdab85bc29007b0c88a4e2153b6f84ec5327
 Key Length: 64 characters
 Secret Key: 5a041cabf258ed6c15d4b04fd8d9cdab...

 Step 2: Retrieve title for pub_0.txt...
------------------------------------------------------------
 Requesting title for: pub_0.txt
Response Status Code: 200
Title Retrieved: Fundamental Asynchronous Framework for Edge Computing in Heterogeneous Environment using Simulation and Quantum Computing

 RESULTS
Filename: pub_0.txt
Title: Fundamental Asynchronous Framework for Edge Computing in Heterogeneous Environment using Simulation and Quantum Computing
First Word: Fundamental

 SUCCESS! Title retrieval is working!


In [5]:
print("=" * 60)
print("TESTING MULTIPLE TITLE RETRIEVALS")
print("=" * 60)

# Test retrieving first 5 publications
test_files = [f"pub_{i}.txt" for i in range(5)]

print(f"\n Testing with {len(test_files)} files: {test_files}")
print("\n" + "-" * 60)

results = []

for filename in test_files:
    title = get_publication_title(secret_key, filename)
    
    if title:
        first_word = extract_first_word(title)
        results.append({
            'filename': filename,
            'title': title,
            'first_word': first_word
        })
        print(f"{filename}: '{first_word}'")
    else:
        print(f"{filename}: Failed")
    
    # Small delay to avoid rate limiting
    time.sleep(0.02)

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Successfully retrieved: {len(results)}/{len(test_files)} titles")

if results:
    print("\nFirst words extracted:")
    for r in results:
        print(f"  {r['filename']}: {r['first_word']}")
    
    # Count frequency
    from collections import Counter
    word_counts = Counter(r['first_word'] for r in results)
    print(f"\nWord frequencies: {dict(word_counts)}")
    

TESTING MULTIPLE TITLE RETRIEVALS

 Testing with 5 files: ['pub_0.txt', 'pub_1.txt', 'pub_2.txt', 'pub_3.txt', 'pub_4.txt']

------------------------------------------------------------
 Requesting title for: pub_0.txt
Response Status Code: 200
Title Retrieved: Fundamental Asynchronous Framework for Edge Computing in Heterogeneous Environment using Simulation and Quantum Computing
pub_0.txt: 'Fundamental'
 Requesting title for: pub_1.txt
Response Status Code: 200
Title Retrieved: Experimental Stochastic Analysis for Computer Vision in Stochastic Environment using Infrastructure and Blockchain Technology
pub_1.txt: 'Experimental'
 Requesting title for: pub_2.txt
Response Status Code: 200
Title Retrieved: Linear Asynchronous Architecture for Quantum Computing in Adaptive Environment using Interface and Blockchain Technology
pub_2.txt: 'Linear'
 Requesting title for: pub_3.txt
Response Status Code: 200
Title Retrieved: Distributed Bayesian Architecture for Internet of Things in Bayesian E

In [6]:
from multiprocessing.dummy import Pool
from multiprocessing import cpu_count
from collections import Counter
import time

# -------------------------------
# Mapper Function
# -------------------------------
def mapper(file_chunk):
    local_counter = Counter()

    # Each worker logs in separately
    secret_key = login(STUDENT_ID)
    if not secret_key:
        print("Worker login failed!")
        return local_counter

    for filename in file_chunk:
        retries = 3

        while retries > 0:
            title = get_publication_title(secret_key, filename)

            if title:
                first_word = extract_first_word(title)
                if first_word:
                    local_counter[first_word] += 1
                break

            retries -= 1
            time.sleep(0.05)  # retry backoff

        # Rate limiting control
        time.sleep(0.015)

    return local_counter


# -------------------------------
# Chunking Function
# -------------------------------
def create_chunks(file_list, n_chunks):
    chunk_size = len(file_list) // n_chunks
    chunks = []

    for i in range(n_chunks):
        start = i * chunk_size
        end = None if i == n_chunks - 1 else (i + 1) * chunk_size
        chunks.append(file_list[start:end])

    return chunks


# -------------------------------
# Main Function
# -------------------------------
def main():
    print("=" * 60)
    print("STARTING PARALLEL PROCESSING")
    print("=" * 60)

    # Step 1: Generate filenames
    files = [f"pub_{i}.txt" for i in range(1000)]

    # Step 2: Workers (safe for Mac + API)
    num_workers = min(6, cpu_count())
    print(f"Using {num_workers} workers")

    # Step 3: Create chunks
    chunks = create_chunks(files, num_workers)

    # Step 4: Multiprocessing
    with Pool(processes=num_workers) as pool:
        results = pool.map(mapper, chunks)

    # Step 5: Reduce
    final_counter = Counter()
    for c in results:
        final_counter.update(c)

    # Step 6: Top 10
    top_10 = [word for word, _ in final_counter.most_common(10)]

    print("\n" + "=" * 60)
    print("TOP 10 WORDS")
    print("=" * 60)
    print(top_10)

    return top_10

In [7]:
def verify_solution(top_10):
    print("\n" + "=" * 60)
    print("VERIFYING SOLUTION")
    print("=" * 60)

    # Fresh login
    secret_key = login(STUDENT_ID)
    if not secret_key:
        print("Verification login failed!")
        return

    verify_url = f"{BASE_URL}/verify"

    payload = {
        "secret_key": secret_key,
        "top_10": top_10
    }

    try:
        response = requests.post(verify_url, json=payload)

        print(f"Status Code: {response.status_code}")

        if response.status_code == 200:
            data = response.json()
            print("\nVerification Result:")
            print(json.dumps(data, indent=4))
        else:
            print("Verification failed!")
            print(response.text)

    except Exception as e:
        print(f"Error: {e}")

In [ ]:
if __name__ == "__main__":
    from multiprocessing import set_start_method

    try:
        set_start_method("spawn") 
    except RuntimeError:
        pass

    top_10_words = main()
    verify_solution(top_10_words)

STARTING PARALLEL PROCESSING
Using 6 workers
 Attempting login for student: MDS202506
Sending POST request to: http://72.60.221.150:8080/login
 Payload: {'student_id': 'MDS202506'}
------------------------------------------------------------
 Attempting login for student: MDS202506
Sending POST request to: http://72.60.221.150:8080/login
 Payload: {'student_id': 'MDS202506'}
------------------------------------------------------------
 Attempting login for student: MDS202506
Sending POST request to: http://72.60.221.150:8080/login
 Payload: {'student_id': 'MDS202506'}
------------------------------------------------------------
 Attempting login for student: MDS202506
Sending POST request to: http://72.60.221.150:8080/login
 Payload: {'student_id': 'MDS202506'}
------------------------------------------------------------
 Attempting login for student: MDS202506
Sending POST request to: http://72.60.221.150:8080/login
 Payload: {'student_id': 'MDS202506'}
-------------------------------